In [1]:
import pandas as pd
import numpy as np
from google_play_scraper import reviews
import tensorflow as tf

print("ready")

ready


In [2]:
from google_play_scraper import reviews, Sort, search
import pandas as pd
import random
import time
import re

keywords = ["food delivery", "food", "restaurant"]

all_data = []
seen_apps = set()
seen_reviews = set()

target_apps = 5
reviews_per_app = 1200

print("Target apps:", target_apps)


def clean_review(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_valid_review(text):
    if len(text.split()) < 4:   
        return False
    if len(text) > 250:         
        return False
    return True

# تحديد sentiment (Binary)
def get_sentiment(r):
    if r >= 4:
        return "positive"
    elif r <= 2:
        return "negative"
    else:
        return None   

while len(seen_apps) < target_apps:

    key = random.choice(keywords)
    print("\nSearching:", key)

    try:
        apps = search(key, lang='en', country='us', n_hits=50)
        random.shuffle(apps)

        for app in apps:

            if len(seen_apps) >= target_apps:
                break

            app_id = app['appId']
            app_name = app['title']

            if app_id in seen_apps:
                continue

            seen_apps.add(app_id)

            print("Scraping:", app_name)

            collected = 0
            continuation_token = None

            try:
                while collected < reviews_per_app:

                    result, continuation_token = reviews(
                        app_id,
                        lang='en',
                        country='us',
                        sort=Sort.NEWEST,
                        count=200,
                        continuation_token=continuation_token
                    )

                    if not result:
                        break

                    for r in result:

                        text = r['content']
                        if not text:
                            continue

                        text = clean_review(text)

                        if not is_valid_review(text):
                            continue

                        if text in seen_reviews:
                            continue

                        sentiment = get_sentiment(r['score'])

                        if sentiment is None:
                            continue  # حذف neutral

                        seen_reviews.add(text)

                        all_data.append({
                            "app_name": app_name,
                            "review": text,
                            "rating": r['score'],
                            "sentiment": sentiment
                        })

                        collected += 1

                        if collected >= reviews_per_app:
                            break

                    if continuation_token is None:
                        break

                    time.sleep(1)

            except:
                continue

    except:
        continue


df = pd.DataFrame(all_data)


df.drop_duplicates(subset=['review'], inplace=True)


min_class = df['sentiment'].value_counts().min()

df_balanced = pd.concat([
    df[df['sentiment'] == 'positive'].sample(min_class, random_state=42),
    df[df['sentiment'] == 'negative'].sample(min_class, random_state=42)
])

df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)


df_balanced.to_csv("clean_reviews_binary.csv", index=False)

print("\nDone")
print("Apps:", len(seen_apps))
print("Final Reviews:", len(df_balanced))
print(df_balanced['sentiment'].value_counts())

Target apps: 5

Searching: food delivery
Scraping: EatSure Food Delivery
Scraping: DoorDash - Dasher
Scraping: Bolt Food: Delivery & Takeaway
Scraping: HelloFresh: Meal Kit Delivery
Scraping: Grubhub: Food Delivery

Done
Apps: 5
Final Reviews: 4266
sentiment
positive    2133
negative    2133
Name: count, dtype: int64


In [3]:
df = df[df['sentiment'] != 'neutral']

label_map = {
    "negative": 0,
    "positive": 1
}

df['label'] = df['sentiment'].map(label_map)

In [4]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_text)
df = df[df['clean_review'] != ""]

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=20000)
tokenizer.fit_on_texts(df['clean_review'])

sequences = tokenizer.texts_to_sequences(df['clean_review'])

X = pad_sequences(sequences, maxlen=100)
y = df['label']

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

model = Sequential()

model.add(Embedding(input_dim=20000, output_dim=128))

model.add(Bidirectional(
    LSTM(64, dropout=0.2, recurrent_dropout=0.2)
))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid')) 
model.compile(
    loss='binary_crossentropy', 
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [10]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=2
)

Epoch 1/10
150/150 - 22s - 147ms/step - accuracy: 0.7956 - loss: 0.4403 - val_accuracy: 0.9042 - val_loss: 0.2604
Epoch 2/10
150/150 - 15s - 102ms/step - accuracy: 0.9333 - loss: 0.1972 - val_accuracy: 0.9058 - val_loss: 0.2750
Epoch 3/10
150/150 - 15s - 101ms/step - accuracy: 0.9542 - loss: 0.1273 - val_accuracy: 0.8933 - val_loss: 0.3066


In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)
print("Test Loss:", loss)

38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9042 - loss: 0.2604
Test Accuracy: 0.9041666388511658
Test Loss: 0.26038241386413574


In [12]:
def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=100)
    
    pred = model.predict(padded)[0][0]

    if pred >= 0.5:
        return "Positive"
    else:
        return "Negative"

In [13]:
print(predict_sentiment("This app is amazing and very useful"))
print(predict_sentiment("This app is useless and waste of time"))
print(predict_sentiment("I like it but sometimes it crashes"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 807ms/step
Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Positive


In [14]:
sample_texts = df['review'].sample(10, random_state=42)

for text in sample_texts:
    print(text)
    print("➡", predict_sentiment(text))
    print("-" * 50)

great way to make money watch out for scam contact support
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
➡ Negative
--------------------------------------------------
can get very expensive if you dont pick meals carefullyi do worry in the summer how cold the food is staying during shipping some times the ice packs are almost melted upon delivery
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
➡ Negative
--------------------------------------------------
worst application of food delivery order time with other time slot but both time eatsure has cancelled my order after let me wait for hrnow i m chasing for my refund from days pls dont try this food delivery application
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
➡ Negative
--------------------------------------------------
im a gold member in they only let me dash in one area support tells me they cant schedule for no but if the app not working what do we suppose to do i work very hard to become a gold
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
➡ Negative
-------

In [15]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred_classes))
print(classification_report(y_test, y_pred_classes))

38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step  
[[713  64]
 [ 51 372]]
              precision    recall  f1-score   support

           0       0.93      0.92      0.93       777
           1       0.85      0.88      0.87       423

    accuracy                           0.90      1200
   macro avg       0.89      0.90      0.90      1200
weighted avg       0.91      0.90      0.90      1200

